In [ ]:
from scipy.io import loadmat, savemat
import time
import os
import urllib.request as ur
import io
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
from tqdm import tqdm

baseURL = "http://ieeg-swez.ethz.ch/long-term_dataset/"
basePath = "/mnt/DATA/NonLinearMI/iEEG"
appendix="info.mat"


# Download raw data

Download the first available hour without seizures (only missing files, about 11 GB in total)

In [ ]:
fp = open(os.path.join(basePath,"dataSources.txt"), "w")
for i in range(1,19):
    out_fname=f"iEEG_ID{i:02}.mat"
    out_path = os.path.join(basePath, out_fname)
    sub = f"ID{i:02}"
    fname = "_".join([sub, appendix])
    URL = os.path.join(baseURL, sub, fname)
    request_data = ur.urlopen(URL)
    file_like = io.BytesIO(request_data.read())
    mat = loadmat(file_like)
    bad_hours = set()
    for p in zip(mat['seizure_begin'], mat['seizure_end']):
        for e in p:
            bad_hours.add(int(e[0]/3600))
    if i in [15,17]:
        bad_hours.add(0)

    for j in range(100):
        if j not in bad_hours:
            data_fname = "_".join([sub, f"{j+1}h.mat"])
            mat_path = os.path.join(baseURL, sub, data_fname)
            print(i, mat_path)
            print(i, mat_path, bad_hours, file=fp)
            if not os.path.isfile(out_path):
                print(ur.urlretrieve(mat_path, out_path)[0])
            break
fp.close()

In [ ]:
out_fname=f"iEEG_info.csv"
out_path = os.path.join(basePath, out_fname)

if not os.path.isfile(out_path):
    frequencies=[]
    for i in range(1,19):
        sub = f"ID{i:02}"
        fname = "_".join([sub, appendix])
        URL = os.path.join(baseURL, sub, fname)
        request_data = ur.urlopen(URL)
        file_like = io.BytesIO(request_data.read())
        mat = loadmat(file_like)
        frequencies.append([i,mat['fs'][0,0]])
    subj_data = pd.DataFrame(frequencies, columns=["Subject", "Hz"]).set_index("Subject")
    subj_data.to_csv(out_path, index=True)
else:
    subj_data = pd.read_csv(out_path, index_col=0)

display(subj_data)


# Data overview

In [ ]:
out_fname=f"iEEG_info.csv"
out_path = os.path.join(basePath, out_fname)
if "Electrodes" not in subj_data:
    electrodes = []
    samples = []
    for i in range(1,19):
        out_fname=f"iEEG_ID{i:02}.mat"
        in_path = os.path.join(basePath, out_fname)
        mat = loadmat(in_path)
        print(i, mat['EEG'].shape)
        electrodes.append(mat['EEG'].shape[0])
        samples.append(mat['EEG'].shape[1])
    subj_data["Electrodes"]=electrodes
    subj_data["Samples"]=samples
    subj_data.to_csv(out_path, index=True)

subj_data.hist("Electrodes", bins="auto")
plt.title(subj_data.Electrodes.max())
plt.show()

# Downsampling

## The three methods are equivalent

In [ ]:
pID = 1
out_fname=f"iEEG_ID{pID:02}.mat"
in_path = os.path.join(basePath, out_fname)
mat = loadmat(in_path)
plt.plot(mat['EEG'][0,:512], label="Orig")

resampled, t = signal.resample(mat['EEG'][0,:512], 100, np.arange(512))
plt.plot(t, resampled, ".", label="Resampl")

decimated = signal.decimate(mat['EEG'][0,:512], 5)
plt.plot(5*np.arange(len(decimated)), decimated, "x", label="Decim")

# sampling frequency
f_sample = 512
# pass band frequency
f_pass = 44
# stop band frequency
f_stop = 70
# pass band ripple
g_pass = 0.5
# stop band attenuation
g_stop = 8.1
N, Wn = signal.buttord(f_pass, f_stop, g_pass, g_stop, analog=False, fs=f_sample)
print(N, Wn)
sos = signal.butter(N, Wn, 'lp', False, 'sos', f_sample) 
#double pass Butterworth
filtered = signal.sosfilt(sos, mat['EEG'][0,:512])
filtered = signal.sosfilt(sos, filtered[::-1])[::-1]
filter_resample, t = signal.resample(filtered, 100, np.arange(512))

plt.plot(filtered, ls="-.", lw=1, label="Butt")
plt.plot(np.arange(len(filtered),step=5),filtered[::5], "+k", label="Butt Skip")
plt.plot(t, filter_resample, "3y", label="Butt Res")
plt.legend()
plt.show()
# plt.plot(np.arange(512)-256,np.fft.fftshift(np.fft.fft(mat['EEG'][0,:512]))/4, label="Original")
# plt.plot(np.arange(512)-256,np.fft.fftshift(np.fft.fft(filtered))/4, label="Butterworth")
plt.plot(np.arange(100)-50,np.fft.fftshift(np.fft.fft(resampled)), label="Resampled")
decl = len(decimated)
plt.plot(np.arange(decl)-int(decl/2),np.fft.fftshift(np.fft.fft(decimated)), label="Decimated")
plt.plot(np.arange(decl)-int(decl/2),np.fft.fftshift(np.fft.fft(filtered[::5])), label="Butt skipping")
plt.plot(np.arange(100)-50,np.fft.fftshift(np.fft.fft(filter_resample)), label="Butt Resamp")
plt.ylim((-300, 300))
plt.legend()
plt.show()

In [ ]:
bands=[[ 1.,  4.], [ 4.,  8.], [ 8., 12.], [12., 15.], [15., 18.], [18., 30.], [30., 44.], [12., 30.], [ 1., 40.]]


N=4
selected_electrodes = {}
if os.path.isfile(os.path.join(basePath,"usedElectrodes.txt")):
    with open(os.path.join(basePath,"usedElectrodes.txt"), "w") as fp:
        for i in tqdm(range(1,19)):
            selected_electrodes[i] = np.sort(np.random.choice(subj_data.loc[i, "Electrodes"], 24, False))
            print(i, selected_electrodes[i], file=fp)
else:
    with open(os.path.join(basePath,"usedElectrodes.txt"), "r") as fp:
        for line in fp:
            bla = line.replace("["," ").replace("]"," ").split()
            selected_electrodes[int(bla[0])]=np.array([int(a) for a in bla[1:]])
            assert len(selected_electrodes[int(bla[0])])==24
            
fs_dest = 100
tot_samp_dest = 10000
sec_dest = int(tot_samp_dest / fs_dest)+1
tot_sec = 3600

new_dataset = np.empty([tot_samp_dest, 24, 18])
for j, band in enumerate(bands, start=1):#
    for i in tqdm(range(1,19), desc=f"Band {j}"):
        f_sample = subj_data.loc[i, "Hz"]

        sos = signal.butter(N, band, 'bp', False, 'sos', f_sample)
        tot_samp_orig = f_sample * (sec_dest+1) + 100
        out_fname=f"iEEG_ID{i:02}.mat"
        in_path = os.path.join(basePath, out_fname)
        
        EEG = loadmat(in_path)['EEG']
        filtered = signal.sosfilt(sos, signal.sosfilt(sos, EEG[selected_electrodes[i],:tot_samp_orig])[:,::-1])[:,-100:f_sample:-1]
        new_dataset[:,:,i-1]=signal.resample(filtered.T, sec_dest*fs_dest)[:tot_samp_dest,:]
    savemat(f"/mnt/DATA/NonLinearMI/iEEG/iEEG_part_band{j}.mat",{"EEG":new_dataset})#

In [ ]:
bands=[[ 1.,  4.], [ 4.,  8.], [ 8., 12.], [12., 15.], [15., 18.], [18., 30.], [30., 44.], [12., 30.], [ 1., 40.]]


N=4
selected_electrodes = {}
if os.path.isfile(os.path.join(basePath,"usedElectrodes.txt")):
    with open(os.path.join(basePath,"usedElectrodes.txt"), "w") as fp:
        for i in tqdm(range(1,19)):
            selected_electrodes[i] = np.sort(np.random.choice(subj_data.loc[i, "Electrodes"], 24, False))
            print(i, selected_electrodes[i], file=fp)
else:
    with open(os.path.join(basePath,"usedElectrodes.txt"), "r") as fp:
        for line in fp:
            bla = line.replace("["," ").replace("]"," ").split()
            selected_electrodes[int(bla[0])]=np.array([int(a) for a in bla[1:]])
            assert len(selected_electrodes[int(bla[0])])==24
            
fs_dest = 100
tot_samp_dest = 10000
sec_dest = int(tot_samp_dest / fs_dest)+1
tot_sec = 3600

new_dataset = np.empty([tot_samp_dest, 24, 18])
for j, band in enumerate([50], start=1):
    for i in tqdm(range(1,19), desc=f"Band {j}"):
        f_sample = subj_data.loc[i, "Hz"]

        sos = signal.butter(N, band, 'lp', False, 'sos', f_sample)
        tot_samp_orig = f_sample * (sec_dest+1) + 100
        out_fname=f"iEEG_ID{i:02}.mat"
        in_path = os.path.join(basePath, out_fname)
        
        EEG = loadmat(in_path)['EEG']
        filtered = signal.sosfilt(sos, signal.sosfilt(sos, EEG[selected_electrodes[i],:tot_samp_orig])[:,::-1])[:,-100:f_sample:-1]
        new_dataset[:,:,i-1]=signal.resample(filtered.T, sec_dest*fs_dest)[:tot_samp_dest,:]
    savemat(f"/mnt/DATA/NonLinearMI/iEEG/iEEG_part.mat",{"EEG":new_dataset})

# Looking for a subject for the sliding window

In [ ]:
subj_data = pd.read_csv("/mnt/DATA/NonLinearMI/iEEG/iEEG_info.csv").set_index("Subject")
seizureDuration=[]
for i in range(1,19):
    out_fname=f"iEEG_ID{i:02}.mat"
    out_path = os.path.join(basePath, out_fname)
    sub = f"ID{i:02}"
    fname = "_".join([sub, appendix])
    URL = os.path.join(baseURL, sub, fname)
    request_data = ur.urlopen(URL)
    file_like = io.BytesIO(request_data.read())
    mat = loadmat(file_like)
    print("Subject:", i, "Electrodes:",subj_data.loc[i,"Electrodes"], "Frequency:", subj_data.loc[i,"Hz"], "Hz")
    for p in zip(mat['seizure_begin'], mat['seizure_end']):
        sd = p[1][0]-p[0][0]
        seizureDuration.append(sd)
        if sd<180:
            continue
        if p[0][0]<1800:
            print("*", end="")
        print(int(sd//60),round(sd%60,2),sep=":")

In [ ]:
plt.hist(seizureDuration, bins="auto")
plt.yscale("log")
plt.show()

In [ ]:
i=12
out_fname=f"iEEG_ID{i:02}.mat"
out_path = os.path.join(basePath, out_fname)
sub = f"ID{i:02}"
fname = "_".join([sub, appendix])
URL = os.path.join(baseURL, sub, fname)
request_data = ur.urlopen(URL)
file_like = io.BytesIO(request_data.read())
mat = loadmat(file_like)
print("Subject:", i, "Electrodes:",subj_data.loc[i,"Electrodes"], "Frequency:", subj_data.loc[i,"Hz"], "Hz")
for p in zip(mat['seizure_begin'], mat['seizure_end']):
    sd = p[1][0]-p[0][0]
    seizureDuration.append(sd)
    if sd<180:
        continue
    if p[0][0]<1800:
        print("*", end="")
    print(int(sd//60),round(sd%60,2),sep=":")
    print(int(p[0][0]//3600),"°:",int((p[0][0]%3600)//60),"' (", int((p[0][0]%3600))," - ", int((p[0][0]%3600)//15),") ",int(p[1][0]//3600),"°:",int((p[1][0]%3600)//60),"' (", int((p[1][0]%3600))," - ", int((p[1][0]%3600)//15),")", sep="")

In [ ]:
i=12
for h in [174,175,176]:
    out_fname=f"iEEG_ID{i:02}.mat"
    out_path = os.path.join(basePath, out_fname)
    sub = f"ID{i:02}"
    appendix=f"{h}h.mat"
    fname = "_".join([sub, appendix])
    if os.path.isfile(os.path.join(basePath, fname)):
        mat = loadmat(os.path.join(basePath, fname))
    else:
        URL = os.path.join(baseURL, sub, fname)
        request_data = ur.urlopen(URL)
        file_like = io.BytesIO(request_data.read())
        mat = loadmat(file_like)
        savemat(os.path.join(basePath, fname),mat)
    for j in range(0,56,5):
        plt.plot(mat["EEG"][j,::100]+20*j, alpha=0.4, lw=1)
    plt.show()


In [ ]:
bands=[[ 1.,  4.], [ 4.,  8.], [ 8., 12.], [12., 15.], [15., 18.], [18., 30.], [30., 44.], [12., 30.], [ 1., 40.]]


N=4
subject = 12
f_sample = subj_data.loc[subject, "Hz"]


for j, band in tqdm(enumerate(bands, start=1), desc="Band "):
    fs_dest = int(band[1]*2*1.125+0.5)
    tot_samp_dest = 60 * fs_dest
    slide = 15 * fs_dest
    
    sos = signal.butter(N, band, 'bp', False, 'sos', f_sample)
    out_fname=f"ID{subject:02}_176h.mat"
    in_path = os.path.join(basePath, out_fname)
    
    EEG = loadmat(in_path)['EEG']

    filtered = signal.sosfilt(sos, signal.sosfilt(sos, EEG[:,:])[:,::-1])[:,::-1]
    resampled = signal.resample(filtered.T, 3600*fs_dest)
    new_dataset = np.empty([tot_samp_dest, 56, 237])
    for i in range(237):
        new_dataset[:,:,i]=resampled[i*slide:i*slide+tot_samp_dest,:]
    savemat(f"/mnt/DATA/NonLinearMI/iEEG/iEEG_ID12_band{j}.mat",{"EEG":new_dataset})

In [ ]:
for i in range(1,8):
    mat = loadmat(f"/mnt/DATA/NonLinearMI/iEEG/iEEG_ID12_band{i}.mat")["EEG"]
    x=np.arange(mat.shape[0])/mat.shape[0]*60
    plt.plot(x, mat[:,0,115]+100*i-100, alpha=0.5)
# plt.xlim((0,10))
plt.show()

In [ ]:
import json


In [ ]:
for i in range(1,10):
    if not os.path.isfile(f"/mnt/DATA/NonLinearMI/iEEG_ID12_band{i}_bin8/globalStats.json"):
        break
    with open(f"/mnt/DATA/NonLinearMI/iEEG_ID12_band{i}_bin8/globalStats.json") as fp:
        globalStats={k:np.array(v) for k,v in json.load(fp).items()}
    print(i)
    x=np.arange(len(globalStats["globaltotalMI"]))*15
    plt.plot(x, globalStats["globaltotalMI"], label="Total")
    plt.plot(x, globalStats["globalgaussMI"], label="Gauss")
    ylim=plt.ylim()
    plt.plot([1740,1740], ylim, ":k")
    plt.plot([118*15,118*15], ylim, "--k")
    plt.plot([1924,1924], ylim, ":k")
    plt.ylim(ylim)
    plt.xlim((1500,2300))
    plt.legend()
    plt.show()
    plt.plot(x, (globalStats["globaltotalMI"]-globalStats["globalgaussMI"])/globalStats["globaltotalMI"], label="Relative")
    ylim=plt.ylim()
    plt.plot([1740,1740], ylim, ":k")
    plt.plot([118*15,118*15], ylim, "--k")
    plt.plot([1924,1924], ylim, ":k")
    plt.ylim(ylim)
    plt.xlim((1500,2300))
    plt.legend()
    plt.show()

In [ ]:
globalStats.keys()